# MART_P05_CUSTO_PROCEDIMENTO

**P05. Qual é a variação do custo médio do mesmo procedimento hospitalar entre os
hospitais?** *(dashboard, auditoria de custos)*

- Custo médio por internação = valor total ÷ internações, por procedimento × hospital.
- `VL_CUSTO_MEDIO_ESTADO` = custo médio do procedimento considerando todos os hospitais.
- `PCT_VARIACAO` = quanto o hospital está acima (+) ou abaixo (−) da média estadual.
- Para evitar distorção por poucos casos, entram apenas pares com **pelo menos 10
  internações** e procedimentos realizados em **pelo menos 2 hospitais**.

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO MART_P05_CUSTO_PROCEDIMENTO
    (COD_PROCEDIMENTO, NOME_PROCEDIMENTO, COD_CNES, NOME_ESTABELECIMENTO, QTD_INTERNACOES,
     VL_CUSTO_MEDIO, VL_CUSTO_MEDIO_ESTADO, PCT_VARIACAO)
WITH base AS (
    SELECT SK_PROCEDIMENTO, SK_ESTABELECIMENTO,
           SUM(QTD_INTERNACAO) AS QTD, SUM(VL_TOTAL) AS VL
    FROM FATO_INTERNACAO
    GROUP BY SK_PROCEDIMENTO, SK_ESTABELECIMENTO
    HAVING SUM(QTD_INTERNACAO) >= 10
),
estado AS (
    SELECT SK_PROCEDIMENTO, SUM(VL) / SUM(QTD) AS MEDIA
    FROM base
    GROUP BY SK_PROCEDIMENTO
    HAVING COUNT(*) >= 2
)
SELECT p.COD_PROCEDIMENTO, p.NOME_PROCEDIMENTO, e.COD_CNES, e.NOME_ESTABELECIMENTO, b.QTD,
       ROUND(b.VL / b.QTD, 2),
       ROUND(x.MEDIA, 2),
       ROUND(100 * (b.VL / b.QTD - x.MEDIA) / NULLIF(x.MEDIA, 0), 2)
FROM base b
JOIN estado x              ON x.SK_PROCEDIMENTO = b.SK_PROCEDIMENTO
JOIN DIM_PROCEDIMENTO p    ON p.SK_PROCEDIMENTO = b.SK_PROCEDIMENTO
JOIN DIM_ESTABELECIMENTO e ON e.SK_ESTABELECIMENTO = b.SK_ESTABELECIMENTO
"""

In [ ]:
recarregar("MART_P05_CUSTO_PROCEDIMENTO", SQL, origem="FATO_INTERNACAO")

In [ ]:
consultar("SELECT * FROM MART_P05_CUSTO_PROCEDIMENTO FETCH FIRST 10 ROWS ONLY")